In [ ]:
from openai import OpenAI
from pinecone import Pinecone
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index = pc.Index("agent-rag")



In [2]:
def get_query_dense_embedding(query):

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    )

    return response.data[0].embedding
def get_query_sparse_embedding(query):

    response = pc.inference.embed(
        model="pinecone-sparse-english-v0",
        inputs=[query],
        parameters={
            "input_type": "query"
        }
    )

    sparse = response.data[0]

    return {
        "indices": sparse["sparse_indices"],
        "values": sparse["sparse_values"]
    }

In [10]:
def hybrid_search(query, top_k=5):

    # -------------------------
    # Dense Embedding
    # -------------------------

    dense_vector = get_query_dense_embedding(query)

    # -------------------------
    # Sparse Embedding
    # -------------------------

    # sparse_vector = get_query_sparse_embedding(query)

    # -------------------------
    # Hybrid Query
    # -------------------------

    results = index.query(

        vector=dense_vector,

        # sparse_vector=sparse_vector,

        top_k=top_k,

        include_metadata=True

    )

    return results

In [14]:
query = "what is llm"

results = hybrid_search(query)
print(results)

QueryResponse(matches=[ScoredVector(id='Understanding_Climate_Change.pdf_127', score=0.209262848, values=[], metadata={'captions': [], 'chunk_id': 127, 'doc_items': ['#/texts/295'], 'filename': 'Understanding_Climate_Change.pdf', 'headings': ['Land Rights and Protection'], 'origin_binary_hash': '8037439857228326505', 'origin_filename': 'Understanding_Climate_Change.pdf', 'origin_mimetype': 'application/pdf', 'page_numbers': ['20'], 'text': 'Land Rights and Protection\nSecuring land rights for indigenous and local communities is essential for climate justice. Recognizing and protecting these rights ensures that communities can manage their lands sustainably and resist exploitation. Legal frameworks and international agreements must uphold the rights of indigenous peoples.', 'title': 'Land Rights and Protection', 'token_count': 51}), ScoredVector(id='Understanding_Climate_Change.pdf_82', score=0.177578926, values=[], metadata={'captions': [], 'chunk_id': 82, 'doc_items': ['#/texts/183'],

In [15]:
results.matches

[ScoredVector(id='Understanding_Climate_Change.pdf_127', score=0.209262848, values=[], metadata={'captions': [], 'chunk_id': 127, 'doc_items': ['#/texts/295'], 'filename': 'Understanding_Climate_Change.pdf', 'headings': ['Land Rights and Protection'], 'origin_binary_hash': '8037439857228326505', 'origin_filename': 'Understanding_Climate_Change.pdf', 'origin_mimetype': 'application/pdf', 'page_numbers': ['20'], 'text': 'Land Rights and Protection\nSecuring land rights for indigenous and local communities is essential for climate justice. Recognizing and protecting these rights ensures that communities can manage their lands sustainably and resist exploitation. Legal frameworks and international agreements must uphold the rights of indigenous peoples.', 'title': 'Land Rights and Protection', 'token_count': 51}),
 ScoredVector(id='Understanding_Climate_Change.pdf_82', score=0.177578926, values=[], metadata={'captions': [], 'chunk_id': 82, 'doc_items': ['#/texts/183'], 'filename': 'Underst

## rank-bm25

In [16]:
pip install rank-bm25

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import json

chunk_file = r"data/chunks/all_chunks.json"

with open(chunk_file, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Total chunks:", len(chunks))

Total chunks: 183


In [ ]:
from rank_bm25 import BM25Okapi

documents = []
metadatas = []

for chunk in chunks:

    text = chunk["text"]

    documents.append(text)

    metadatas.append(chunk["metadata"])

# Tokenize
tokenized_docs = [
    doc.lower().split()
    for doc in documents
]

# Build BM25 index
bm25 = BM25Okapi(tokenized_docs)

print("BM25 model ready")

BM25 model ready


In [23]:
bm25

In [19]:
def bm25_search(query, top_k=5):

    tokenized_query = query.lower().split()

    scores = bm25.get_scores(tokenized_query)

    # Get top indices
    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]

    results = []

    for idx in top_indices:

        results.append({
            "score": scores[idx],
            "text": documents[idx],
            "metadata": metadatas[idx]
        })

    return results

In [20]:
query = "economic resilience climate adaptation"

results = bm25_search(query, top_k=5)

In [24]:
for result in results:
    print(result)
    print("\n====================")

    print("Score:", result["score"])

    print("Title:",
          result["metadata"].get("title"))

    print("Filename:",
          result["metadata"].get("filename"))

    print("\nText:\n")

    print(result["text"])

{'score': np.float64(11.863136967425719), 'text': 'Economic Resilience\nBuilding economic resilience involves creating diverse and sustainable economies that can withstand climate impacts. This includes supporting small businesses, fostering innovation, and promoting local economic development. Resilient economies are better equipped to adapt to changing conditions and recover from disruptions.', 'metadata': {'filename': 'Understanding_Climate_Change.pdf', 'chunk_id': 136, 'page_numbers': [21], 'title': 'Economic Resilience', 'headings': ['Economic Resilience'], 'captions': [], 'doc_items': ['#/texts/317'], 'token_count': 51, 'origin': {'mimetype': 'application/pdf', 'binary_hash': 8037439857228326505, 'filename': 'Understanding_Climate_Change.pdf', 'uri': None}}}

Score: 11.863136967425719
Title: Economic Resilience
Filename: Understanding_Climate_Change.pdf

Text:

Economic Resilience
Building economic resilience involves creating diverse and sustainable economies that can withstand 

## sparse + Dense

In [25]:
def dense_search(query, top_k=10):

    dense_vector = get_query_dense_embedding(query)

    results = index.query(
        vector=dense_vector,
        top_k=top_k,
        include_metadata=True
    )

    formatted = []

    for rank, match in enumerate(results["matches"]):

        formatted.append({
            "id": match["id"],
            "rank": rank + 1,
            "score": match["score"],
            "text": match["metadata"]["text"],
            "metadata": match["metadata"]
        })

    return formatted
def bm25_search(query, top_k=10):

    tokenized_query = query.lower().split()

    scores = bm25.get_scores(tokenized_query)

    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]

    results = []

    for rank, idx in enumerate(top_indices):

        metadata = metadatas[idx]

        unique_id = (
            f"{metadata['filename']}_"
            f"{metadata['chunk_id']}"
        )

        results.append({
            "id": unique_id,
            "rank": rank + 1,
            "score": scores[idx],
            "text": documents[idx],
            "metadata": metadata
        })

    return results

In [26]:
from collections import defaultdict

def reciprocal_rank_fusion(
    dense_results,
    bm25_results,
    k=60
):

    fused_scores = defaultdict(float)

    chunk_data = {}

    # Dense contribution
    for result in dense_results:

        doc_id = result["id"]

        rank = result["rank"]

        fused_scores[doc_id] += 1 / (k + rank)

        chunk_data[doc_id] = result

    # BM25 contribution
    for result in bm25_results:

        doc_id = result["id"]

        rank = result["rank"]

        fused_scores[doc_id] += 1 / (k + rank)

        chunk_data[doc_id] = result

    # Sort by fused score
    ranked = sorted(
        fused_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    final_results = []

    for doc_id, score in ranked:

        item = chunk_data[doc_id]

        item["fused_score"] = score

        final_results.append(item)

    return final_results

In [27]:
query = "economic resilience climate adaptation"

dense_results = dense_search(query)

bm25_results = bm25_search(query)

final_results = reciprocal_rank_fusion(
    dense_results,
    bm25_results
)

In [28]:
for result in final_results[:5]:

    print("\n====================")

    print("Fused Score:",
          result["fused_score"])

    print("Title:",
          result["metadata"]["title"])

    print(result["text"])


Fused Score: 0.03278688524590164
Title: Economic Resilience
Economic Resilience
Building economic resilience involves creating diverse and sustainable economies that can withstand climate impacts. This includes supporting small businesses, fostering innovation, and promoting local economic development. Resilient economies are better equipped to adapt to changing conditions and recover from disruptions.

Fused Score: 0.030798389007344232
Title: Public Health Infrastructure
Public Health Infrastructure
Strengthening public health infrastructure is vital for adapting to climate change. This includes enhancing healthcare facilities, improving disease surveillance systems, and training healthcare professionals. Community health programs can increase resilience and preparedness for climate-related health risks.

Fused Score: 0.030798389007344232
Title: Economic Impacts of Climate Change
Economic Impacts of Climate Change
The economic costs of climate change include damage to infrastructure,

## metadata filter

In [29]:
available_documents = [
    "NIPS-2012-imagenet-classification-with-deep-convolutional-neural-networks-Paper.pdf",
    "Understanding_Climate_Change.pdf",
    "paper1.pdf"
]

In [30]:
def detect_document_filters(query, documents):

    query_lower = query.lower()

    matched_docs = []

    for doc in documents:

        doc_name = doc.lower()

        # Remove extension
        simplified = (
            doc_name
            .replace(".pdf", "")
            .replace("-", " ")
        )

        if any(
            word in query_lower
            for word in simplified.split()
        ):
            matched_docs.append(doc)

    return matched_docs

In [ ]:
def build_pinecone_filter(matched_docs):

    if not matched_docs:
        return None

    return {
        "filename": {
            "$in": matched_docs
        }
    }
